In [1]:
# ── STEP 0: Memory-allocator tuning (must run before the first CUDA call) ──
# Reduces VRAM fragmentation, which is one of the most common silent causes
# of "OOM" on T4s even when mem_get_info() shows free memory available.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── STEP 1: Force-reinstall PyTorch with CUDA 12.1 (matches Kaggle's current driver) ──
!pip install -q --upgrade torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"GPU count: {n_gpus}")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  [cuda:{i}] {props.name} — {props.total_memory/1e9:.1f} GB")
    x = torch.zeros(3, 3).cuda()
    x.fill_(1.0)
    print("✅ CUDA kernel test passed!")
    if n_gpus < 2:
        print("⚠️  Only 1 GPU detected. For best results: Notebook Settings → Accelerator → 'GPU T4 x2'.")
else:
    print("❌ No GPU — make sure Accelerator is set to GPU in notebook settings")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 68.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 92.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 34.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 6.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 9.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.6 MB/s eta 0:00:0000:0100:01
     ━━━━

In [2]:
import subprocess, sys

def pip_install(*packages: str, label: str = ""):
    """Run pip install and FAIL LOUDLY instead of silently continuing.

    A bare `!pip install` in a notebook cell does NOT stop execution if it
    fails — pip prints an error and the notebook just moves on to the next
    cell. That's exactly how an unsatisfiable version pin (e.g. requesting
    peft>=0.12.0 while colpali-engine==0.3.1 requires peft<0.12.0) turns
    into a confusing `ModuleNotFoundError: No module named 'colpali_engine'`
    several cells later instead of a clear error at install time.
    """
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f"pip install failed{f' ({label})' if label else ''} — see output above")
    print(f"✅ {label or ', '.join(packages)}")

# ── STEP 2: System deps ──
#!apt-get install -y poppler-utils 2>/dev/null | tail -3
!apt-get update -qq
!apt-get install -y poppler-utils
!which pdftoppm pdfinfo
# ── STEP 3: Core packages ──
pip_install(
    "fastapi==0.115.6",
    "uvicorn[standard]==0.34.0",
    "python-multipart==0.0.20",
    "pyngrok==7.2.2",
    "nest-asyncio==1.6.0",
    "Pillow==11.1.0",
    "pdf2image==1.17.0",
    "openpyxl==3.1.5",
    "pandas==2.2.3",
    "python-docx==1.1.2",
    "matplotlib",
    label="core packages",
)

# ── ChromaDB as the vector database ──
# We pass embedding_function=None so chromadb never downloads
# sentence-transformers or onnxruntime — ColPali supplies all vectors.
pip_install("chromadb>=0.5.0", label="chromadb")

# NOTE: "byaldi" was removed here. It was being installed but never used
# (RAG_MODEL was always set to None and nothing referenced it) — it only
# added install time and an extra, unused copy of torch/transformers deps.

print("✅ All dependencies installed!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpoppler-dev libpoppler-private-dev libpoppler118
The following NEW packages will be installed:
  poppler-utils
The following packages will be upgraded:
  libpoppler-dev libpoppler-private-dev libpoppler118
3 upgraded, 1 newly installed, 0 to remove and 180 not upgraded.
Need to get 1,471 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpoppler-private-dev amd64 22.02.0-2ubuntu0.13 [198 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpoppler-dev amd64 22.02.0-2ubuntu0.13 [5,186 B]
Get:3 http://archive.ubuntu.com/ubuntu jammy-u

In [3]:
# ── Pin dependencies to avoid architecture conflicts ──
#
# IMPORTANT: colpali-engine==0.3.1's packaging metadata declares
# numpy<2.0.0, peft<0.12.0, and pillow<11.0.0 — but none of those bounds
# are actually exercised by the ColPali / ColPaliProcessor classes we use
# (no numpy import anywhere in the package; peft is only imported by its
# *training* module, which we never touch; Pillow usage is limited to
# stable, version-agnostic APIs). Letting pip enforce those bounds anyway
# is actively harmful here: Kaggle's image ships pandas pre-built against
# numpy>=2.0, and force-downgrading numpy to satisfy colpali-engine breaks
# pandas's own compiled C extensions with:
#   "ValueError: numpy.dtype size changed, may indicate binary incompatibility"
#
# Fix: install colpali-engine with --no-deps so its metadata constraints
# are never enforced, and don't pin numpy at all — leave Kaggle's
# numpy/pandas pairing exactly as it ships.
#
# bitsandbytes is what lets us load Qwen2.5-VL-7B in 4-bit (NF4) instead
# of full fp16/bf16 — the main fix for the T4 OOM.
pip_install(
    "transformers>=4.49.0,<5.0.0",
    "peft>=0.11.0",
    "accelerate>=0.27.0",
    "bitsandbytes>=0.45.0",
    label="pinned deps (incl. bitsandbytes)",
)
pip_install(
    "colpali-engine==0.3.1",
    "--no-deps",
    label="colpali-engine (--no-deps, see note above)",
)

✅ pinned deps (incl. bitsandbytes)
✅ colpali-engine (--no-deps, see note above)


In [4]:
import os, json, uuid, time, shutil, base64, asyncio, logging
from pathlib import Path
from typing import Optional, List, Dict, Any
from io import BytesIO
from datetime import datetime

import nest_asyncio
nest_asyncio.apply()

import torch
import pandas as pd
from PIL import Image
from pdf2image import convert_from_path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
import uvicorn
from pyngrok import ngrok

# ── Logging ──
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger("BuildMarshalAI")

# ── Paths ──
BASE_DIR   = Path("/kaggle/working/buildmarshal")
DOCS_DIR   = BASE_DIR / "documents"
PAGES_DIR  = BASE_DIR / "pages"
INDEX_DIR  = BASE_DIR / "index"
CHROMA_DIR = BASE_DIR / "chroma_db"    # ChromaDB persistence directory
META_FILE  = BASE_DIR / "metadata.json"

for d in [BASE_DIR, DOCS_DIR, PAGES_DIR, INDEX_DIR, CHROMA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── API Keys via Kaggle Secrets ──
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = secrets.get_secret("NGROK_AUTH_TOKEN")
    logger.info("✅ Loaded secrets from Kaggle Secrets")
except Exception as e:
    logger.warning(f"Kaggle Secrets unavailable ({e}). Falling back to env vars.")
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "3BwpQ66W9cFPYqdLDef3C44WRcS_2nPxX3XjmbhNzRn77otbC")

assert NGROK_AUTH_TOKEN != "YOUR_NGROK_TOKEN_HERE", "❌ Set your NGROK_AUTH_TOKEN!"

# ── Metadata helpers ──
def load_metadata() -> Dict:
    if META_FILE.exists():
        return json.loads(META_FILE.read_text())
    return {"documents": {}}

def save_metadata(meta: Dict):
    META_FILE.write_text(json.dumps(meta, indent=2, default=str))

# =======================================================================
# GPU topology — decides which model goes on which card.
#
#   2x T4 (16GB each — Kaggle's "GPU T4 x2")
#     -> ColPali  on cuda:0  (retrieval, ~6.5GB in fp16)
#     -> Qwen-VL  on cuda:1  (generation, ~5GB in 4-bit) — it gets a whole
#                            GPU to itself, so we also raise max_pixels
#                            later for sharper document/drawing reading.
#
#   1 GPU (fallback — e.g. "GPU P100", or if only 1 T4 is visible)
#     -> Both models share cuda:0. Qwen is still 4-bit quantized, so the
#        combined footprint (~11-12GB) comfortably fits in 16GB.
# =======================================================================
NUM_GPUS           = torch.cuda.device_count()
COLPALI_DEVICE_IDX  = 0
VL_DEVICE_IDX       = 1 if NUM_GPUS >= 2 else 0
COLPALI_DEVICE      = f"cuda:{COLPALI_DEVICE_IDX}"
VL_DEVICE           = f"cuda:{VL_DEVICE_IDX}"
DUAL_GPU            = NUM_GPUS >= 2

print(f"✅ Config ready | GPUs detected: {NUM_GPUS}")
for i in range(NUM_GPUS):
    free, total = torch.cuda.mem_get_info(i)
    print(f"   [cuda:{i}] {torch.cuda.get_device_name(i)} | {free/1e9:.1f} / {total/1e9:.1f} GB free")
if DUAL_GPU:
    print(f"   → ColPali will load on cuda:{COLPALI_DEVICE_IDX}, Qwen2.5-VL on cuda:{VL_DEVICE_IDX}")
else:
    print(f"   → Single GPU detected — ColPali and 4-bit Qwen2.5-VL will share cuda:{VL_DEVICE_IDX}")

2026-07-01 05:59:33,324 | WARNING | Kaggle Secrets unavailable (Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 125424834 and label NGROK_AUTH_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.). Falling back to env vars.


✅ Config ready | GPUs detected: 2
   [cuda:0] Tesla T4 | 15.5 / 15.6 GB free
   [cuda:1] Tesla T4 | 15.5 / 15.6 GB free
   → ColPali will load on cuda:0, Qwen2.5-VL on cuda:1


In [ ]:
# =======================================================================
# CELL 5 — ColPali  +  ChromaDB vector store
# =======================================================================

import gc, torch, numpy as np
import chromadb
from typing import Optional, List, Dict
from PIL import Image

# ── PEFT / transformers KeyError: 'llava' patch ──
# PaliGemma uses a llava-style architecture. When transformers' PEFT integration
# tries to look it up in _MOE_TARGET_MODULE_MAPPING it fails with KeyError: 'llava'.
# These two patches make that lookup a safe no-op.
import transformers.integrations.peft as _peft_integ

if hasattr(_peft_integ, "_convert_peft_config_moe"):
    _orig_moe = _peft_integ._convert_peft_config_moe
    def _safe_moe(p, m):
        try: return _orig_moe(p, m)
        except KeyError: return p
    _peft_integ._convert_peft_config_moe = _safe_moe

if hasattr(_peft_integ, "convert_peft_config_for_transformers"):
    _orig_cvt = _peft_integ.convert_peft_config_for_transformers
    def _safe_cvt(p, model=None, conversions=None):
        try: return _orig_cvt(p, model=model, conversions=conversions)
        except KeyError: return p
    _peft_integ.convert_peft_config_for_transformers = _safe_cvt

print("✅ PEFT patch applied (or not needed on this transformers version)")

# ── Free VRAM before loading ──
for _v in ['COLPALI_MODEL', 'COLPALI_PROCESSOR']:
    if _v in dir() and eval(_v) is not None:
        exec(f"del {_v}")
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM on {COLPALI_DEVICE} before load: {torch.cuda.mem_get_info(COLPALI_DEVICE_IDX)[0]/1e9:.2f} GB")

# ── PaliGemma 'language_model' compatibility patch ──
# colpali_engine==0.3.1 does `model.language_model._tied_weights_keys`,
# but newer transformers wraps PaliGemma's language model under
# `.model.language_model` instead of exposing `.language_model` directly.
from transformers import PaliGemmaForConditionalGeneration

def _colpali_compat_language_model(self):
    direct = self._modules.get("language_model")
    if direct is not None:
        return direct
    return self.model.language_model

PaliGemmaForConditionalGeneration.language_model = property(_colpali_compat_language_model)
print("✅ PaliGemma compatibility patch applied")

# ── Load ColPali using colpali_engine's own classes ──
from colpali_engine.models import ColPali, ColPaliProcessor

ADAPTER_ID = "vidore/colpali-v1.2"

logger.info(f"Loading ColPali v1.2 onto {COLPALI_DEVICE} (fp16)...")
COLPALI_MODEL = ColPali.from_pretrained(
    ADAPTER_ID,
    torch_dtype=torch.float16,            # T4 (Turing) has no native bf16 tensor cores — fp16 is the fast path
    device_map={"": COLPALI_DEVICE_IDX},  # pin explicitly so it never drifts onto Qwen's GPU
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
).eval()

COLPALI_PROCESSOR = ColPaliProcessor.from_pretrained(ADAPTER_ID)

print(f"✅ ColPali ready on {COLPALI_DEVICE}. Free VRAM: {torch.cuda.mem_get_info(COLPALI_DEVICE_IDX)[0]/1e9:.2f} GB")

# =======================================================================
# ChromaDB — persistent vector store with per-page metadata
# =======================================================================

_COLLECTION_NAME = "document_pages"

CHROMA_CLIENT = chromadb.PersistentClient(path=str(CHROMA_DIR))

# ChromaDB >=0.6.x requires an explicit EmbeddingFunction object; passing
# None raises a type error.  We supply a no-op stub because ColPali
# produces all embeddings externally — ChromaDB never calls this function.
class _NoOpEF(chromadb.EmbeddingFunction):
    """Stub: ColPali supplies all vectors; ChromaDB never calls this."""
    def __call__(self, input):
        raise RuntimeError("_NoOpEF should never be called directly")

_NO_OP_EF = _NoOpEF()

try:
    CHROMA_COLLECTION = CHROMA_CLIENT.get_collection(
        name=_COLLECTION_NAME,
        embedding_function=_NO_OP_EF,
    )
    logger.info(
        f"✅ Loaded existing ChromaDB collection "
        f"({CHROMA_COLLECTION.count()} pages indexed)"
    )
except Exception:
    CHROMA_COLLECTION = CHROMA_CLIENT.create_collection(
        name=_COLLECTION_NAME,
        embedding_function=_NO_OP_EF,
        metadata={"hnsw:space": "cosine"},
    )
    logger.info("✅ Created new ChromaDB collection")


@torch.no_grad()
def embed_image(image_path: str) -> Optional[np.ndarray]:
    try:
        img = Image.open(image_path).convert("RGB")

        # process_images() builds the correct token structure for page screenshots
        batch = COLPALI_PROCESSOR.process_images([img]).to(COLPALI_MODEL.device)

        # ColPali.forward() returns a raw Tensor (batch, seq_len, dim) — NOT a ModelOutput
        out = COLPALI_MODEL(**batch)           # shape: (1, seq_len, embedding_dim)
        vec = out.mean(dim=1).squeeze(0)       # mean-pool → (embedding_dim,)

        return vec.float().cpu().numpy()
    except Exception as e:
        logger.error(f"embed_image failed for {image_path}: {e}")
        return None


@torch.no_grad()
def embed_query(query: str) -> Optional[np.ndarray]:
    try:
        # process_queries() builds the correct token structure for text queries
        batch = COLPALI_PROCESSOR.process_queries([query]).to(COLPALI_MODEL.device)

        # Same — ColPali returns a raw Tensor
        out = COLPALI_MODEL(**batch)           # shape: (1, seq_len, embedding_dim)
        vec = out.mean(dim=1).squeeze(0)       # mean-pool → (embedding_dim,)

        return vec.float().cpu().numpy()
    except Exception as e:
        logger.error(f"embed_query failed: {e}")
        return None


def retrieve_context(query: str, top_k: int = 5) -> List[Dict]:
    if CHROMA_COLLECTION.count() == 0:
        logger.warning("ChromaDB collection empty — falling back to all pages")
        return get_all_pages(top_k)

    q_vec = embed_query(query)
    if q_vec is None:
        logger.error("Query embedding failed — falling back to all pages")
        return get_all_pages(top_k)

    n = min(top_k, CHROMA_COLLECTION.count())
    results = CHROMA_COLLECTION.query(
        query_embeddings=[q_vec.tolist()],
        n_results=n,
        include=["metadatas", "distances"],
    )

    pages = []
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        similarity = round(1.0 - dist, 4)
        pages.append({
            "doc_name":     meta["doc_name"],
            "doc_id":       meta["doc_id"],
            "page":         int(meta["page_num"]),
            "image_path":   meta.get("image_path", ""),
            "text_content": meta.get("text_content", ""),
            "score":        similarity,
        })

    if pages:
        logger.info(f"Retrieved {len(pages)} pages (top similarity: {pages[0]['score']})")
    return pages


def get_all_pages(limit: int = 5) -> List[Dict]:
    meta, pages = load_metadata(), []
    for doc_id, doc in meta["documents"].items():
        for page in doc["pages"]:
            pages.append({
                "doc_name":     doc["name"],
                "doc_id":       doc_id,
                "page":         page["page_num"],
                "image_path":   page.get("image_path", ""),
                "text_content": page.get("text_content", ""),
                "score":        0.0,
            })
            if len(pages) >= limit:
                return pages
    return pages


print("✅ ColPali + ChromaDB vector store ready")
print(f"   Collection '{_COLLECTION_NAME}' — {CHROMA_COLLECTION.count()} pages indexed")

✅ PEFT patch applied (or not needed on this transformers version)
Free VRAM on cuda:0 before load: 15.51 GB


2026-07-01 05:59:48.594781: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782885589.037346      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782885589.157427      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782885590.177599      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782885590.177640      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782885590.177643      58 computation_placer.cc:177] computation placer alr

✅ PaliGemma compatibility patch applied


adapter_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/862M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of the model checkpoint at vidore/colpaligemma-3b-pt-448-base were not used when initializing ColPali: ['model.language_model.model.embed_tokens.weight', 'model.language_model.model.layers.0.input_layernorm.weight', 'model.language_model.model.layers.0.mlp.down_proj.weight', 'model.language_model.model.layers.0.mlp.gate_proj.weight', 'model.language_model.model.layers.0.mlp.up_proj.weight', 'model.language_model.model.layers.0.post_attention_layernorm.weight', 'model.language_model.model.layers.0.self_attn.k_proj.weight', 'model.language_model.model.layers.0.self_attn.o_proj.weight', 'model.language_model.model.layers.0.self_attn.q_proj.weight', 'model.language_model.model.layers.0.self_attn.v_proj.weight', 'model.language_model.model.layers.1.input_layernorm.weight', 'model.language_model.model.layers.1.mlp.down_proj.weight', 'model.language_model.model.layers.1.mlp.gate_proj.weight', 'model.language_model.model.layers.1.mlp.up_proj.weight', 'model.language_model.model.la

adapter_model.safetensors:   0%|          | 0.00/78.6M [00:00<?, ?B/s]

In [ ]:
# =======================================================================
# CELL 6 — Document Ingestion  ->  ColPali embedding  ->  ChromaDB
# =======================================================================
#
# Flow for each file:
#   convert to page screenshots
#   for each page:
#       embed_image(screenshot)  -> float32 numpy vector
#       CHROMA_COLLECTION.add(id, embedding, metadata, document)
#   persist metadata.json (for /api/documents listing and deletion)
# =======================================================================

def process_pdf(file_path: Path, doc_id: str) -> List[Dict]:
    logger.info(f"Processing PDF: {file_path.name}")
    pages = convert_from_path(str(file_path), dpi=150, fmt='png')
    page_infos = []
    for i, page_img in enumerate(pages):
        page_path = PAGES_DIR / f"{doc_id}_page_{i+1}.png"
        page_img.thumbnail((1024, 1024))
        page_img.save(str(page_path), "PNG")
        page_infos.append({
            "page_num":    i + 1,
            "image_path":  str(page_path),
            "width":       page_img.width,
            "height":      page_img.height,
            "text_content": "",
        })
    logger.info(f"PDF: {len(pages)} pages extracted")
    return page_infos


def _df_to_image(df, title, out_path):
    fig, ax = plt.subplots(figsize=(min(20, len(df.columns)*1.5+1),
                                    min(20, len(df)*0.4+1)))
    ax.axis('off')
    data = df.head(50).fillna('').astype(str)
    tbl  = ax.table(cellText=data.values, colLabels=data.columns,
                    cellLoc='left', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.3)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#4472C4')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#f0f0f0' if r % 2 == 0 else 'white')
    plt.title(title, fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(out_path), dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def process_excel(file_path: Path, doc_id: str) -> List[Dict]:
    page_infos = []
    dfs = ({"Sheet1": pd.read_csv(str(file_path))}
           if file_path.suffix == '.csv'
           else pd.read_excel(str(file_path), sheet_name=None))
    for sheet_name, df in dfs.items():
        page_path = PAGES_DIR / f"{doc_id}_sheet_{sheet_name}.png"
        try:
            _df_to_image(df, f"{file_path.stem} — {sheet_name}", page_path)
            page_infos.append({
                "page_num":    len(page_infos) + 1,
                "image_path":  str(page_path),
                "sheet_name":  sheet_name,
                "text_content": df.head(100).to_string(),
            })
        except Exception as e:
            logger.warning(f"Sheet render failed: {e}")
            page_infos.append({
                "page_num":    len(page_infos) + 1,
                "image_path":  None,
                "sheet_name":  sheet_name,
                "text_content": df.head(100).to_string(),
            })
    return page_infos


def process_image(file_path: Path, doc_id: str) -> List[Dict]:
    img = Image.open(str(file_path)).convert("RGB")
    img.thumbnail((1024, 1024))
    page_path = PAGES_DIR / f"{doc_id}_img.png"
    img.save(str(page_path), "PNG")
    return [{
        "page_num": 1, "image_path": str(page_path),
        "width": img.width, "height": img.height, "text_content": "",
    }]


def process_docx(file_path: Path, doc_id: str) -> List[Dict]:
    from docx import Document as DocxDocument
    doc       = DocxDocument(str(file_path))
    full_text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    page_infos = []
    chunks = [full_text[i:i+2000] for i in range(0, max(len(full_text), 1), 2000)]
    for idx, chunk in enumerate(chunks):
        page_path = PAGES_DIR / f"{doc_id}_page_{idx+1}.png"
        try:
            fig, ax = plt.subplots(figsize=(10, 14)); ax.axis('off')
            ax.text(0.05, 0.95, chunk, transform=ax.transAxes, fontsize=9,
                    verticalalignment='top', fontfamily='monospace', wrap=True)
            plt.title(f"{file_path.stem} — Page {idx+1}")
            plt.savefig(str(page_path), dpi=120, bbox_inches='tight', facecolor='white')
            plt.close(fig)
            page_infos.append({
                "page_num":    idx + 1,
                "image_path":  str(page_path),
                "text_content": chunk,
            })
        except Exception as e:
            page_infos.append({
                "page_num":    idx + 1,
                "image_path":  None,
                "text_content": chunk,
            })
    return page_infos
def process_audio(file_path: Path, doc_id: str) -> List[Dict]:
    """
    Audio files can't be visually embedded; store as a single 'page'.
    The pages endpoint will serve the raw audio file for playback.
    """
    # Copy audio to pages dir for serving
    page_path = PAGES_DIR / f"{doc_id}_audio{file_path.suffix}"
    shutil.copy2(str(file_path), str(page_path))
    
    return [{
        "page_num": 1,
        "image_path": str(page_path),  # repurpose image_path for serving
        "text_content": f"Audio file: {file_path.name}",
    }]


def process_text(file_path: Path, doc_id: str) -> List[Dict]:
    """Process plain text files (txt, json, xml, code files etc.)."""
    text = file_path.read_text(errors='replace')
    page_infos = []
    chunks = [text[i:i+3000] for i in range(0, max(len(text), 1), 3000)]
    
    for idx, chunk in enumerate(chunks):
        page_path = PAGES_DIR / f"{doc_id}_page_{idx+1}.png"
        try:
            fig, ax = plt.subplots(figsize=(10, 14)); ax.axis('off')
            ax.text(0.05, 0.95, chunk, transform=ax.transAxes, fontsize=8,
                    verticalalignment='top', fontfamily='monospace', wrap=True)
            plt.title(f"{file_path.stem} — Page {idx+1}")
            plt.savefig(str(page_path), dpi=120, bbox_inches='tight', facecolor='white')
            plt.close(fig)
            page_infos.append({
                "page_num": idx + 1,
                "image_path": str(page_path),
                "text_content": chunk,
            })
        except Exception:
            page_infos.append({
                "page_num": idx + 1,
                "image_path": None,
                "text_content": chunk,
            })
    return page_infos


def process_generic(file_path: Path, doc_id: str) -> List[Dict]:
    """Fallback — store file metadata only."""
    page_path = PAGES_DIR / f"{doc_id}_file{file_path.suffix}"
    shutil.copy2(str(file_path), str(page_path))
    return [{
        "page_num": 1,
        "image_path": str(page_path),
        "text_content": f"File: {file_path.name} ({file_path.stat().st_size} bytes)",
    }]


def ingest_document(file_path: Path, doc_id: str) -> Dict:
    """
    Convert document to page screenshots, embed each with ColPali,
    and store the resulting vector + metadata in ChromaDB.

    ChromaDB entry per page
      id        : "<doc_id>_p<page_num>"  unique string key
      embedding : ColPali mean-pooled float32 vector (provided as list)
      metadata  : doc_id, doc_name, page_num, image_path, text_content
      document  : text_content (ChromaDB full-text field)
    """
    ext  = file_path.suffix.lower()
    meta = load_metadata()

    if ext == '.pdf':
        pages = process_pdf(file_path, doc_id)
    elif ext in ('.xlsx', '.xls', '.csv'):
        pages = process_excel(file_path, doc_id)
    elif ext in ('.png', '.jpg', '.jpeg', '.gif', '.webp', '.bmp', '.tiff'):
        pages = process_image(file_path, doc_id)
    elif ext in ('.doc', '.docx'):
        pages = process_docx(file_path, doc_id)
    elif ext in ('.mp3', '.wav', '.ogg', '.m4a', '.flac', '.aac', '.wma', '.opus'):
        pages = process_audio(file_path, doc_id)
    elif ext in ('.txt', '.csv', '.json', '.xml', '.html', '.css', '.js', '.py', 
                 '.java', '.c', '.cpp', '.md', '.rtf'):
        pages = process_text(file_path, doc_id)
    else:
        pages = process_generic(file_path, doc_id)  # fallback instead of error

    logger.info(f"Embedding {len(pages)} pages and storing in ChromaDB...")
    embedded = 0

    for page in pages:
        img_path = page.get("image_path")
        if not img_path or not os.path.exists(img_path):
            logger.warning(f"Skipping page {page['page_num']} — no image file")
            continue

        # 1. Embed the page screenshot with ColPali
        embedding = embed_image(img_path)
        if embedding is None:
            logger.warning(f"Embedding returned None for page {page['page_num']}")
            continue

        # 2. Build ChromaDB entry
        chroma_id  = f"{doc_id}_p{page['page_num']}"
        # ChromaDB metadata values must be str / int / float / bool.
        # Truncate text_content to 1000 chars to stay within metadata limits.
        text_trunc = page.get("text_content", "")[:1000]

        page_metadata = {
            "doc_id":       doc_id,
            "doc_name":     file_path.name,
            "page_num":     int(page["page_num"]),
            "image_path":   img_path,
            "text_content": text_trunc,
        }

        # 3. Upsert: delete existing entry (no-op if absent), then add fresh
        try:
            CHROMA_COLLECTION.delete(ids=[chroma_id])
        except Exception:
            pass

        CHROMA_COLLECTION.add(
            ids=[chroma_id],
            embeddings=[embedding.tolist()],   # ChromaDB requires a Python list
            metadatas=[page_metadata],
            documents=[text_trunc],
        )

        embedded += 1
        torch.cuda.empty_cache()   # release VRAM after every forward pass

    logger.info(
        f"✅ ChromaDB: {embedded}/{len(pages)} pages stored for '{file_path.name}' "
        f"| Collection total: {CHROMA_COLLECTION.count()} pages"
    )

    doc_meta = {
        "id":         doc_id,
        "name":       file_path.name,
        "type":       ext,
        "size":       file_path.stat().st_size,
        "pages":      pages,
        "page_count": len(pages),
        "status":     "indexed",
        "created_at": datetime.now().isoformat(),
    }
    meta["documents"][doc_id] = doc_meta
    save_metadata(meta)
    return doc_meta


print("✅ Ingestion pipeline ready — ColPali embeddings stored in ChromaDB")

In [ ]:
# =======================================================================
# CELL 7 — VRAM Manager  +  Qwen2.5-VL-7B response generation (4-bit)
# =======================================================================

import gc, torch, os, asyncio, base64
from typing import Optional, List, Dict

def free_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    for i in range(NUM_GPUS):
        free = torch.cuda.mem_get_info(i)[0] / 1e9
        logger.info(f"VRAM cuda:{i} after cleanup: {free:.2f} GB free")
    return torch.cuda.mem_get_info(COLPALI_DEVICE_IDX)[0] / 1e9

def offload_colpali():
    """Manual safety valve: move ColPali to CPU. Only useful in single-GPU
    mode if a particularly large request still runs tight on VRAM — not
    needed (and not called automatically) when running on 2 GPUs, since
    each model already has its own card."""
    global COLPALI_MODEL
    if COLPALI_MODEL is not None:
        try:
            COLPALI_MODEL.to("cpu")
            gc.collect(); torch.cuda.empty_cache()
            logger.info(f"ColPali offloaded to CPU. Free VRAM on {COLPALI_DEVICE}: {torch.cuda.mem_get_info(COLPALI_DEVICE_IDX)[0]/1e9:.2f} GB")
        except Exception as e:
            logger.warning(f"ColPali offload failed: {e}")

def reload_colpali():
    global COLPALI_MODEL
    if COLPALI_MODEL is not None:
        try:
            COLPALI_MODEL.to(COLPALI_DEVICE)
            torch.cuda.synchronize()
            logger.info(f"ColPali reloaded to {COLPALI_DEVICE}. Free VRAM: {torch.cuda.mem_get_info(COLPALI_DEVICE_IDX)[0]/1e9:.2f} GB")
        except Exception as e:
            logger.warning(f"ColPali reload failed: {e}")


pip_install("qwen-vl-utils==0.0.8", label="qwen-vl-utils")

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from PIL import Image

VL_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

if "VL_MODEL" in dir() and VL_MODEL is not None:
    del VL_MODEL
gc.collect(); torch.cuda.empty_cache()
print(f"Free VRAM on {VL_DEVICE} before Qwen load: {torch.cuda.mem_get_info(VL_DEVICE_IDX)[0]/1e9:.2f} GB")

# ── 4-bit NF4 quantization — this is the main OOM fix ──
# Full bf16/fp16 Qwen2.5-VL-7B needs ~16.6GB of weights alone, which is
# already tighter than a single T4's 16GB before counting any KV-cache or
# activation memory. 4-bit (QLoRA-style NF4 + double quant) drops the
# language-model weights to ~4GB. We keep the vision tower ("visual") in
# fp16 via llm_int8_skip_modules — it's small (~0.6B params, ~1.2GB) and
# skipping it preserves document/drawing-reading quality.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 = Turing, no native bf16 tensor cores
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual"],
)

logger.info(f"Loading Qwen2.5-VL-7B onto {VL_DEVICE} (4-bit NF4, vision tower kept in fp16)...")
VL_MODEL = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VL_MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": VL_DEVICE_IDX},   # pin the whole model to its GPU — never split across cards
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
VL_MODEL.eval()

# When Qwen has a dedicated GPU (dual T4), there's headroom to read
# documents at a higher resolution. When it shares a GPU with ColPali,
# stay conservative to leave room for KV-cache + activations.
if DUAL_GPU:
    VL_MIN_PIXELS, VL_MAX_PIXELS = 256 * 28 * 28, 768 * 28 * 28
else:
    VL_MIN_PIXELS, VL_MAX_PIXELS = 256 * 28 * 28, 512 * 28 * 28

VL_PROCESSOR = AutoProcessor.from_pretrained(
    VL_MODEL_ID,
    min_pixels=VL_MIN_PIXELS,
    max_pixels=VL_MAX_PIXELS,
)
VL_QUANT_LABEL = "qwen2.5-vl-7b-bnb-4bit"
print(f"✅ Qwen2.5-VL ready on {VL_DEVICE}! Free VRAM: {torch.cuda.mem_get_info(VL_DEVICE_IDX)[0]/1e9:.2f} GB")


def vl_generate(messages: list, max_new_tokens: int = 1024) -> str:
    text = VL_PROCESSOR.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = VL_PROCESSOR(
        text=[text],
        images=image_inputs if image_inputs else None,
        videos=video_inputs if video_inputs else None,
        padding=True,
        return_tensors="pt",
    ).to(VL_MODEL.device)   # device-agnostic — works whether Qwen is on cuda:0 or cuda:1

    with torch.no_grad():
        output_ids = VL_MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    text_out = VL_PROCESSOR.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    del inputs, output_ids, trimmed
    if not DUAL_GPU:
        gc.collect(); torch.cuda.empty_cache()   # extra cleanup when sharing the GPU with ColPali

    return text_out


def image_to_base64(image_path: str) -> Optional[str]:
    try:
        with open(image_path, "rb") as f:
            data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/png;base64,{data}"
    except Exception:
        return None


async def generate_response(
    query: str,
    context_pages: List[Dict],
    history: List[Dict],
    model: str = "qwen2.5-vl",
) -> Dict:
    system_prompt = (
        "You are BuildMarshalAI, an intelligent construction document assistant.\n"
        "You help users understand and analyze uploaded project documents "
        "(PDFs, spreadsheets, images, drawings).\n"
        "- Answer ONLY based on the provided document context.\n"
        "- If the answer is not in the documents, say so clearly.\n"
        "- Cite the document name and page number for each piece of information.\n"
        "- Use markdown headers, bullet points, and tables where appropriate.\n"
        "- Describe drawings or diagrams in detail if visible."
    )

    messages = [{"role": "system", "content": system_prompt}]

    for msg in history[-6:]:
        messages.append({
            "role":    msg.get("role", "user"),
            "content": msg.get("content", ""),
        })

    user_content = []
    images_added = 0
    for page in context_pages:
        if images_added >= 4:
            break
        if page.get("image_path") and os.path.exists(page["image_path"]):
            user_content.append({"type": "image", "image": page["image_path"]})
            images_added += 1

    context_text = "Relevant document pages:\n"
    for i, page in enumerate(context_pages):
        context_text += (
            f"\n[Source {i+1}: {page['doc_name']}, Page {page['page']}"
            f" | similarity={page.get('score', 0):.3f}]"
        )
        if page.get("text_content"):
            context_text += f"\n{page['text_content'][:800]}"

    user_content.append({
        "type": "text",
        "text": f"{context_text}\n\nUser question: {query}",
    })
    messages.append({"role": "user", "content": user_content})

    try:
        loop     = asyncio.get_event_loop()
        response = await loop.run_in_executor(
            None, lambda: vl_generate(messages, max_new_tokens=1024)
        )

        sources = []
        for page in context_pages:
            source = {
                "doc_name":  page["doc_name"],
                "doc_id":    page["doc_id"],
                "page":      page["page"],
                "score":     page.get("score", 0.0),
                "image_url": None,
            }
            if page.get("image_path"):
                source["image_url"] = image_to_base64(page["image_path"])
            sources.append(source)

        return {"response": response, "sources": sources, "model_used": VL_QUANT_LABEL}

    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise HTTPException(
            status_code=500,
            detail="GPU out of memory. Try a shorter query or fewer documents.",
        )
    except Exception as e:
        logger.error(f"Inference error: {e}")
        raise HTTPException(status_code=500, detail=f"Inference error: {str(e)}")


print("✅ Query pipeline ready — ColPali (ChromaDB retrieval) + Qwen2.5-VL (4-bit generation)")

In [ ]:
# =======================================================================
# CELL 8 — FastAPI application
# =======================================================================

app = FastAPI(
    title="BuildMarshalAI Backend",
    description="ColPali (ChromaDB vector store) + Qwen2.5-VL document chatbot",
    version="2.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/api/health")
async def health_check():
    gpu_info = []
    if torch.cuda.is_available():
        for i in range(NUM_GPUS):
            free, total = torch.cuda.mem_get_info(i)
            hosts = []
            if i == COLPALI_DEVICE_IDX:
                hosts.append("colpali")
            if i == VL_DEVICE_IDX:
                hosts.append("qwen2.5-vl")
            gpu_info.append({
                "index":    i,
                "name":     torch.cuda.get_device_name(i),
                "free_gb":  round(free / 1e9, 2),
                "total_gb": round(total / 1e9, 2),
                "hosts":    hosts,
            })
    return {
        "status":          "healthy",
        "gpu_available":   torch.cuda.is_available(),
        "gpu_count":       NUM_GPUS,
        "gpus":            gpu_info,
        "chroma_pages":    CHROMA_COLLECTION.count(),
        "documents_count": len(load_metadata().get("documents", {})),
    }


@app.get("/api/status")
async def get_status():
    meta  = load_metadata()
    docs  = meta.get("documents", {})
    total = sum(d.get("page_count", 0) for d in docs.values())
    return {
        "documents_count": len(docs),
        "total_pages":     total,
        "chroma_pages":    CHROMA_COLLECTION.count(),
        "retriever":       f"colpali-v1.2 + chromadb ({COLPALI_DEVICE})",
        "generator":       f"{VL_QUANT_LABEL} ({VL_DEVICE})",
        "gpu_count":       NUM_GPUS,
    }


@app.post("/api/upload")
async def upload_document(file: UploadFile = File(...), doc_id: str = Form(None)):
    if not doc_id:
        doc_id = str(uuid.uuid4())[:12]
    ext     = Path(file.filename).suffix.lower()

    allowed = {
    # Documents
    '.pdf', '.xlsx', '.xls', '.csv', '.doc', '.docx', '.pptx', '.ppt',
    '.txt', '.rtf', '.odt', '.ods',
    # Images
    '.png', '.jpg', '.jpeg', '.gif', '.webp', '.bmp', '.tiff',
    # Audio
    '.mp3', '.wav', '.ogg', '.m4a', '.flac', '.aac', '.wma', '.opus',
    # Video
    '.mp4', '.webm', '.mov', '.avi', '.mkv',
    # Archives & Code
    '.zip', '.rar', '.7z', '.tar', '.gz',
    '.json', '.xml', '.html', '.css', '.js', '.py', '.java', '.c', '.cpp', '.md',
}

    if ext not in allowed:
        raise HTTPException(400, detail=f"Unsupported file type: {ext}")
    save_path = DOCS_DIR / f"{doc_id}{ext}"
    try:
        save_path.write_bytes(await file.read())
    except Exception as e:
        raise HTTPException(500, detail=f"Could not save file: {e}")
    try:
        doc_meta = ingest_document(save_path, doc_id)
        return {
            "id":      doc_id,
            "name":    file.filename,
            "pages":   doc_meta["page_count"],
            "status":  "indexed",
            "message": (
                f"Indexed {doc_meta['page_count']} pages into ChromaDB "
                f"(collection total: {CHROMA_COLLECTION.count()})"
            ),
        }
    except Exception as e:
        logger.error(f"Ingestion error: {e}")
        return {"id": doc_id, "name": file.filename, "pages": 0,
                "status": "error", "message": str(e)}


@app.get("/api/documents")
async def list_documents():
    meta = load_metadata()
    return {"documents": [
        {
            "id":         did,
            "name":       d["name"],
            "type":       d["type"],
            "size":       d.get("size", 0),
            "pages":      d.get("page_count", 0),
            "status":     d.get("status", "indexed"),
            "created_at": d.get("created_at"),
        }
        for did, d in meta.get("documents", {}).items()
    ]}


@app.get("/api/documents/{doc_id}/status")
async def document_status(doc_id: str):
    doc = load_metadata().get("documents", {}).get(doc_id)
    if not doc:
        raise HTTPException(404, detail="Document not found")
    return {
        "id":     doc_id,
        "status": doc.get("status", "unknown"),
        "pages":  doc.get("page_count", 0),
    }


@app.delete("/api/documents/{doc_id}")
async def delete_document(doc_id: str):
    meta = load_metadata()
    doc  = meta.get("documents", {}).get(doc_id)
    if not doc:
        raise HTTPException(404, detail="Document not found")

    # 1. Remove page screenshots from disk
    for page in doc.get("pages", []):
        img = page.get("image_path")
        if img and os.path.exists(img):
            os.remove(img)
    for f in DOCS_DIR.glob(f"{doc_id}*"):
        f.unlink()

    # 2. Remove all ChromaDB vectors that belong to this document.
    #    We filter by the "doc_id" metadata field so every page entry
    #    for this document is deleted in a single call.
    try:
        CHROMA_COLLECTION.delete(where={"doc_id": doc_id})
        logger.info(
            f"ChromaDB: removed entries for doc_id={doc_id} "
            f"| Collection now has {CHROMA_COLLECTION.count()} pages"
        )
    except Exception as e:
        logger.warning(f"ChromaDB delete failed for {doc_id}: {e}")

    del meta["documents"][doc_id]
    save_metadata(meta)
    return {"message": "Document deleted", "id": doc_id}


@app.post("/api/chat")
async def chat(request: Request):
    body  = await request.json()
    query = body.get("query", "").strip()
    if not query:
        raise HTTPException(400, detail="Query cannot be empty")

    history = body.get("history", [])
    model   = body.get("model", "qwen2.5-vl")
    top_k   = min(body.get("top_k", 5), 20)

    logger.info(f"Query: {query[:80]}...")

    # Step 1: Retrieve top-k pages with ColPali + ChromaDB
    context_pages = retrieve_context(query, top_k=top_k)
    logger.info(f"Retrieved {len(context_pages)} pages from ChromaDB")

    # Step 2: Generate answer with Qwen2.5-VL using retrieved pages
    return await generate_response(query, context_pages, history, model=model)


@app.get("/api/pages/{doc_id}/{page_num}")
async def get_page_image(doc_id: str, page_num: int):
    doc = load_metadata().get("documents", {}).get(doc_id)
    if not doc:
        raise HTTPException(404, detail="Document not found")
    for page in doc.get("pages", []):
        if page.get("page_num") == page_num:
            img_path = page.get("image_path")
            if img_path and os.path.exists(img_path):
                # Detect content type from extension
                ext = Path(img_path).suffix.lower()
                content_types = {
                    '.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
                    '.gif': 'image/gif', '.webp': 'image/webp',
                    '.mp3': 'audio/mpeg', '.wav': 'audio/wav', '.ogg': 'audio/ogg',
                    '.m4a': 'audio/mp4', '.flac': 'audio/flac', '.aac': 'audio/aac',
                    '.mp4': 'video/mp4', '.webm': 'video/webm', '.mov': 'video/quicktime',
                }
                ct = content_types.get(ext, 'application/octet-stream')
                with open(img_path, "rb") as f:
                    return StreamingResponse(BytesIO(f.read()), media_type=ct)
    raise HTTPException(404, detail="Page not found")



@app.get("/api/chroma/stats")
async def chroma_stats():
    """Inspect ChromaDB collection: total count + sample of stored entries."""
    count  = CHROMA_COLLECTION.count()
    sample = []
    if count > 0:
        peek = CHROMA_COLLECTION.peek(limit=5)
        for i, meta in enumerate(peek.get("metadatas", [])):
            sample.append({
                "id":       peek["ids"][i],
                "doc_name": meta.get("doc_name"),
                "page_num": meta.get("page_num"),
                "doc_id":   meta.get("doc_id"),
            })
    return {"total_pages": count, "sample": sample}

# ═══ Management APIs (Trades, Vendors, Team Members) ═══

MGMT_FILE = BASE_DIR / "management.json"

def load_mgmt() -> Dict:
    if MGMT_FILE.exists():
        return json.loads(MGMT_FILE.read_text())
    return {
        "trades": [
            {"id":"tr-1","name":"Carpentry","description":"-","status":"Active"},
            {"id":"tr-2","name":"Concrete","description":"-","status":"Active"},
            {"id":"tr-3","name":"Electrical","description":"-","status":"Active"},
            {"id":"tr-4","name":"HVAC","description":"-","status":"Active"},
            {"id":"tr-5","name":"Plumbing","description":"-","status":"Active"},
            {"id":"tr-6","name":"Painting","description":"-","status":"Active"},
        ],
        "vendors": [],
        "team_members": [
            {"id":"tm-1","name":"admin","email":"admin@bm.ai","department":"—","category":"internal"},
        ]
    }

def save_mgmt(data: Dict):
    MGMT_FILE.write_text(json.dumps(data, indent=2, default=str))


print("✅ FastAPI app defined — ChromaDB-backed endpoints ready")

In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# CELL 6b — Management APIs (Trades, Vendors, Team Members)
# ═══════════════════════════════════════════════════════════════════════

MGMT_FILE = BASE_DIR / "management.json"

def load_mgmt() -> Dict:
    if MGMT_FILE.exists():
        return json.loads(MGMT_FILE.read_text())
    return {
        "trades": [
            {"id":"tr-1","name":"Carpentry","description":"-","status":"Active"},
            {"id":"tr-2","name":"Concrete","description":"-","status":"Active"},
            {"id":"tr-3","name":"Drywall","description":"-","status":"Active"},
            {"id":"tr-4","name":"Electrical","description":"-","status":"Active"},
            {"id":"tr-5","name":"Exterior Works","description":"-","status":"Active"},
            {"id":"tr-6","name":"Flooring / Finishing","description":"-","status":"Active"},
            {"id":"tr-7","name":"HVAC","description":"-","status":"Active"},
            {"id":"tr-8","name":"Insulation","description":"-","status":"Active"},
            {"id":"tr-9","name":"Landscaping","description":"-","status":"Active"},
            {"id":"tr-10","name":"Masonry","description":"-","status":"Active"},
            {"id":"tr-11","name":"Painting","description":"-","status":"Active"},
            {"id":"tr-12","name":"Plumbing","description":"-","status":"Active"},
            {"id":"tr-13","name":"Roofing","description":"-","status":"Active"},
        ],
        "vendors": [
            {"id":"v-1","name":"Clover Paints","vendorType":"Material Supplier","trade":"Painting","activeProjects":0,"status":"Active"},
            {"id":"v-2","name":"Jim's electrical Company","vendorType":"Material Supplier","trade":"Electrical","activeProjects":1,"status":"Active"},
        ],
        "team_members": [
            {"id":"tm-1","name":"admin","email":"admin@bm.ai","department":"—","category":"internal"},
            {"id":"tm-2","name":"Jim Co","email":"jim@buildmarshal.ai","department":"Finance & Accounts","category":"internal"},
        ]
    }

def save_mgmt(data: Dict):
    MGMT_FILE.write_text(json.dumps(data, indent=2, default=str))


# ── Trades CRUD ──
@app.get("/api/trades")
async def list_trades():
    return {"trades": load_mgmt().get("trades", [])}

@app.post("/api/trades")
async def create_trade(request: Request):
    body = await request.json()
    mgmt = load_mgmt()
    trade = {
        "id": f"tr-{uuid.uuid4().hex[:8]}",
        "name": body.get("name", "").strip(),
        "description": body.get("description", "-"),
        "status": body.get("status", "Active")
    }
    if not trade["name"]:
        raise HTTPException(400, detail="Name is required")
    mgmt["trades"].append(trade)
    save_mgmt(mgmt)
    return trade

@app.put("/api/trades/{trade_id}")
async def update_trade(trade_id: str, request: Request):
    body = await request.json()
    mgmt = load_mgmt()
    for t in mgmt["trades"]:
        if t["id"] == trade_id:
            t["name"] = body.get("name", t["name"])
            t["description"] = body.get("description", t["description"])
            t["status"] = body.get("status", t["status"])
            save_mgmt(mgmt)
            return t
    raise HTTPException(404, detail="Trade not found")

@app.delete("/api/trades/{trade_id}")
async def delete_trade(trade_id: str):
    mgmt = load_mgmt()
    mgmt["trades"] = [t for t in mgmt["trades"] if t["id"] != trade_id]
    save_mgmt(mgmt)
    return {"message": "Trade deleted", "id": trade_id}


# ── Vendors CRUD ──
@app.get("/api/vendors")
async def list_vendors():
    return {"vendors": load_mgmt().get("vendors", [])}

@app.post("/api/vendors")
async def create_vendor(request: Request):
    body = await request.json()
    mgmt = load_mgmt()
    vendor = {
        "id": f"v-{uuid.uuid4().hex[:8]}",
        "name": body.get("name", "").strip(),
        "vendorType": body.get("vendorType", "Material Supplier"),
        "trade": body.get("trade", ""),
        "activeProjects": body.get("activeProjects", 0),
        "status": body.get("status", "Active")
    }
    if not vendor["name"]:
        raise HTTPException(400, detail="Name is required")
    mgmt["vendors"].append(vendor)
    save_mgmt(mgmt)
    return vendor

@app.put("/api/vendors/{vendor_id}")
async def update_vendor(vendor_id: str, request: Request):
    body = await request.json()
    mgmt = load_mgmt()
    for v in mgmt["vendors"]:
        if v["id"] == vendor_id:
            v["name"] = body.get("name", v["name"])
            v["vendorType"] = body.get("vendorType", v["vendorType"])
            v["trade"] = body.get("trade", v["trade"])
            v["status"] = body.get("status", v["status"])
            save_mgmt(mgmt)
            return v
    raise HTTPException(404, detail="Vendor not found")

@app.delete("/api/vendors/{vendor_id}")
async def delete_vendor(vendor_id: str):
    mgmt = load_mgmt()
    mgmt["vendors"] = [v for v in mgmt["vendors"] if v["id"] != vendor_id]
    save_mgmt(mgmt)
    return {"message": "Vendor deleted", "id": vendor_id}


# ── Team Members CRUD ──
@app.get("/api/team-members")
async def list_team_members():
    return {"team_members": load_mgmt().get("team_members", [])}

@app.post("/api/team-members")
async def create_team_member(request: Request):
    body = await request.json()
    mgmt = load_mgmt()
    member = {
        "id": f"tm-{uuid.uuid4().hex[:8]}",
        "name": body.get("name", "").strip(),
        "email": body.get("email", ""),
        "department": body.get("department", "—"),
        "category": body.get("category", "internal"),
        "company": body.get("company", ""),
        "contactName": body.get("contactName", ""),
    }
    if not member["name"]:
        raise HTTPException(400, detail="Name is required")
    mgmt["team_members"].append(member)
    save_mgmt(mgmt)
    return member

@app.put("/api/team-members/{member_id}")
async def update_team_member(member_id: str, request: Request):
    body = await request.json()
    mgmt = load_mgmt()
    for m in mgmt["team_members"]:
        if m["id"] == member_id:
            for key in ["name", "email", "department", "category", "company", "contactName"]:
                if key in body:
                    m[key] = body[key]
            save_mgmt(mgmt)
            return m
    raise HTTPException(404, detail="Team member not found")

@app.delete("/api/team-members/{member_id}")
async def delete_team_member(member_id: str):
    mgmt = load_mgmt()
    mgmt["team_members"] = [m for m in mgmt["team_members"] if m["id"] != member_id]
    save_mgmt(mgmt)
    return {"message": "Team member deleted", "id": member_id}

In [ ]:
def start_server():
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    port = 8001
    import requests
    try:
        resp = requests.get(
            "https://api.ngrok.com/endpoints",
            headers={
                "Authorization": f"Bearer {NGROK_AUTH_TOKEN}",
                "Ngrok-Version": "2",
            },
        )
        resp.raise_for_status()
        for ep in resp.json().get("endpoints", []):
            requests.delete(
                f"https://api.ngrok.com/endpoints/{ep['id']}",
                headers={
                    "Authorization": f"Bearer {NGROK_AUTH_TOKEN}",
                    "Ngrok-Version": "2",
                },
            )
            print(f"🔌 Closed stale remote tunnel: {ep.get('public_url')}")
    except Exception as e:
        print(f"⚠️ Could not query/close remote ngrok endpoints: {e}")

    public_url = ngrok.connect(port)

    print("\n" + "=" * 60)
    print("🏗️  BuildMarshalAI Backend is LIVE!")
    print("=" * 60)
    print(f"\n🌐 PUBLIC URL : {public_url.public_url}")
    print(f"📋 Copy into your frontend Settings panel")
    print(f"🔗 API Docs   : {public_url.public_url}/docs")
    print(f"💚 Health     : {public_url.public_url}/api/health")
    print(f"📊 ChromaDB   : {public_url.public_url}/api/chroma/stats")
    print(f"\n🖥️  GPUs       : {NUM_GPUS} detected | ColPali → {COLPALI_DEVICE} | Qwen2.5-VL → {VL_DEVICE}")
    print("\n" + "=" * 60)
    print("⚠️  URL changes each session. Keep this cell running!")
    print("=" * 60 + "\n")

    uvicorn.run(app, host="0.0.0.0", port=port)

start_server()